# 二階檢索與 Re-Rank 排序機制

這份教學版 notebook 的目標是要**明顯看出**：

1. **第一階段向量檢索**：先把「看起來可能相關」的候選抓回來  
2. **第二階段 Re-Rank**：再把真正最符合 query 細節的內容排到前面  

這份案例特別設計了很多**容易混淆的干擾資料**，所以你通常會看到：

- 向量檢索抓回很多「有蝙蝠俠 / 英雄 / 動作」但不完全符合需求的候選
- Re-Rank 根據 query 的完整語意（例如：**黑暗、嚴肅、寫實、不要搞笑**）重新排序
- 最後再交給 **Groq 的 LLaMA 模型**做生成式回答


In [ ]:
import os
import getpass
import numpy as np
import pandas as pd
from dotenv import load_dotenv

from google.colab import userdata
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

pd.set_option("display.max_colwidth", 120)

load_dotenv()  # 從 .env 檔案載入環境變數


In [ ]:
# 讀取 GROQ API Key
try:
    groq_api_key = os.getenv("GROQ_API_KEY")
except Exception:
    groq_api_key = None

if not groq_api_key:
    groq_api_key = getpass.getpass("請輸入 GROQ_API_KEY：")

os.environ["GROQ_API_KEY"] = groq_api_key
print("GROQ_API_KEY 已設定完成。")


GROQ_API_KEY 已設定完成。


In [4]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("Groq LLaMA 模型載入完成。")


Groq LLaMA 模型載入完成。


## 建立「故意容易混淆」的教學資料集

這裡的資料不是只有真正答案，還刻意加入很多「看起來很像，但其實不夠符合需求」的候選，例如：

- 也是蝙蝠俠，但太搞笑
- 很黑暗，但不算超級英雄動作片
- 是超級英雄，但不是蝙蝠俠
- 有對決、有動作，但整體風格不對

這樣第一階段向量檢索就比較容易混進雜訊，rerank 才能顯示價值。


In [5]:
movies_data = [
    {"title": "黑暗騎士", "content": "蝙蝠俠與小丑的對決，黑暗、寫實、嚴肅的超級英雄動作片，節奏緊湊，充滿道德衝突。"},
    {"title": "Batman Begins", "content": "蝙蝠俠的起源故事，風格沉穩黑暗，偏寫實，帶有訓練、成長與打擊犯罪元素。"},
    {"title": "蝙蝠俠大戰超人", "content": "蝙蝠俠與超人的衝突與對決，黑暗風格的超級英雄動作電影，氣氛嚴肅。"},
    {"title": "樂高蝙蝠俠電影", "content": "以蝙蝠俠為主角的動畫電影，雖然有英雄元素，但整體風格搞笑、歡樂、適合家庭。"},
    {"title": "小丑", "content": "與蝙蝠俠宇宙有關的黑暗劇情片，角色心理描寫強烈，但不是典型超級英雄動作片。"},
    {"title": "閃電俠", "content": "DC 超級英雄電影，節奏快，有動作場面與多重宇宙元素，但主角不是蝙蝠俠。"},
    {"title": "復仇者聯盟", "content": "經典超級英雄動作片，英雄眾多、娛樂性高，但沒有蝙蝠俠，整體風格不算特別黑暗。"},
    {"title": "蜘蛛人：無家日", "content": "多重宇宙英雄冒險，屬於超級英雄動作片，但主角不是蝙蝠俠，風格較熱鬧。"},
    {"title": "名偵探柯南：萬聖節的新娘", "content": "推理動畫電影，有爆炸、追逐與動作元素，但不是超級英雄題材。"},
    {"title": "神偷奶爸", "content": "動畫喜劇，帶有反派與科技元素，風格輕鬆搞笑，不是超級英雄電影。"},
    {"title": "超人：鋼鐵英雄", "content": "超級英雄動作片，帶有較嚴肅風格與大規模戰鬥，但主角是超人，不是蝙蝠俠。"},
    {"title": "守護者", "content": "黑暗、成人向、帶有政治與道德思辨的英雄電影，風格很沉重，但不是蝙蝠俠作品。"}
]

docs = [
    Document(page_content=item["content"], metadata={"title": item["title"]})
    for item in movies_data
]

pd.DataFrame(movies_data)


,title,content
0,黑暗騎士,蝙蝠俠與小丑的對決，黑暗、寫實、嚴肅的超級英雄動作片，節奏緊湊，充滿道德衝突。
1,Batman Begins,蝙蝠俠的起源故事，風格沉穩黑暗，偏寫實，帶有訓練、成長與打擊犯罪元素。
2,蝙蝠俠大戰超人,蝙蝠俠與超人的衝突與對決，黑暗風格的超級英雄動作電影，氣氛嚴肅。
3,樂高蝙蝠俠電影,以蝙蝠俠為主角的動畫電影，雖然有英雄元素，但整體風格搞笑、歡樂、適合家庭。
4,小丑,與蝙蝠俠宇宙有關的黑暗劇情片，角色心理描寫強烈，但不是典型超級英雄動作片。
5,閃電俠,DC 超級英雄電影，節奏快，有動作場面與多重宇宙元素，但主角不是蝙蝠俠。
6,復仇者聯盟,經典超級英雄動作片，英雄眾多、娛樂性高，但沒有蝙蝠俠，整體風格不算特別黑暗。
7,蜘蛛人：無家日,多重宇宙英雄冒險，屬於超級英雄動作片，但主角不是蝙蝠俠，風格較熱鬧。
8,名偵探柯南：萬聖節的新娘,推理動畫電影，有爆炸、追逐與動作元素，但不是超級英雄題材。
9,神偷奶爸,動畫喜劇，帶有反派與科技元素，風格輕鬆搞笑，不是超級英雄電影。


## 建立向量資料庫
這一步負責的是：**把 query 跟文件都轉成向量，先做第一階段召回（recall）**


In [6]:
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = FAISS.from_documents(docs, embeddings)
print("向量資料庫建立完成。")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

向量資料庫建立完成。


## 設定查詢：加入「細條件」

這次不是只問「蝙蝠俠電影」，而是多加幾個限制條件：

- 要 **蝙蝠俠相關**
- 要 **黑暗、嚴肅、寫實**
- **不要搞笑**

這樣向量檢索可能還是會抓到《樂高蝙蝠俠電影》這種「字面很像」的候選，  
但 rerank 理論上應該把它往後壓。


In [7]:
query = "我想看蝙蝠俠相關電影，但要黑暗、嚴肅、寫實，不要搞笑，最好有英雄對決或打擊犯罪的感覺"
print("Query：", query)


Query： 我想看蝙蝠俠相關電影，但要黑暗、嚴肅、寫實，不要搞笑，最好有英雄對決或打擊犯罪的感覺


## 第一階段：向量檢索（Recall）

注意這裡我們**刻意抓比較多候選**，例如 `k=8`。  
因為第一階段的工作不是直接挑冠軍，而是：

> **先把可能相關的都抓回來，寧可多一點，也不要漏掉。**


In [8]:
vector_results = vectorstore.similarity_search_with_score(query, k=8)

vector_rows = []
for rank, (doc, score) in enumerate(vector_results, start=1):
    vector_rows.append({
        "vector_rank": rank,
        "title": doc.metadata["title"],
        "vector_score": float(score),
        "content": doc.page_content
    })

vector_df = pd.DataFrame(vector_rows)
vector_df


,vector_rank,title,vector_score,content
0,1,守護者,0.390483,黑暗、成人向、帶有政治與道德思辨的英雄電影，風格很沉重，但不是蝙蝠俠作品。
1,2,蝙蝠俠大戰超人,0.412785,蝙蝠俠與超人的衝突與對決，黑暗風格的超級英雄動作電影，氣氛嚴肅。
2,3,小丑,0.469989,與蝙蝠俠宇宙有關的黑暗劇情片，角色心理描寫強烈，但不是典型超級英雄動作片。
3,4,黑暗騎士,0.481430,蝙蝠俠與小丑的對決，黑暗、寫實、嚴肅的超級英雄動作片，節奏緊湊，充滿道德衝突。
4,5,Batman Begins,0.504522,蝙蝠俠的起源故事，風格沉穩黑暗，偏寫實，帶有訓練、成長與打擊犯罪元素。
5,6,復仇者聯盟,0.526243,經典超級英雄動作片，英雄眾多、娛樂性高，但沒有蝙蝠俠，整體風格不算特別黑暗。
6,7,蜘蛛人：無家日,0.607120,多重宇宙英雄冒險，屬於超級英雄動作片，但主角不是蝙蝠俠，風格較熱鬧。
7,8,樂高蝙蝠俠電影,0.622011,以蝙蝠俠為主角的動畫電影，雖然有英雄元素，但整體風格搞笑、歡樂、適合家庭。


### 先觀察第一階段結果

請特別注意：

- 有些結果雖然**字面上很接近**
- 但其實不一定符合 query 的完整需求

例如：
- 有蝙蝠俠，但太搞笑
- 很黑暗，但不是超級英雄動作片
- 是超級英雄動作片，但不是蝙蝠俠


## 第二階段：Re-Rank（Precision）

這裡使用 cross-encoder reranker。  
它和 embedding 最大的差別是：

- **embedding**：query 與文件分開編碼，再用向量距離比相似度
- **rerank**：query 與候選文件一起看，再直接判斷「這一對到底多匹配」

因此 rerank 更能處理：

- `不要搞笑`
- `偏黑暗、寫實`
- `最好有英雄對決`
- `雖然都提到蝙蝠俠，但哪個更符合需求`


In [9]:
reranker_model_name = "BAAI/bge-reranker-base"
reranker = CrossEncoder(reranker_model_name)

candidate_docs = [doc for doc, _ in vector_results]
rerank_inputs = [
    [query, f"標題：{doc.metadata['title']}；內容：{doc.page_content}"]
    for doc in candidate_docs
]

rerank_scores = reranker.predict(rerank_inputs)

rerank_rows = []
for doc, vrow, rscore in zip(candidate_docs, vector_rows, rerank_scores):
    rerank_rows.append({
        "title": doc.metadata["title"],
        "vector_rank": vrow["vector_rank"],
        "vector_score": vrow["vector_score"],
        "rerank_score": float(rscore),
        "content": doc.page_content
    })

rerank_df = pd.DataFrame(rerank_rows).sort_values("rerank_score", ascending=False).reset_index(drop=True)
rerank_df["rerank_rank"] = np.arange(1, len(rerank_df) + 1)
rerank_df = rerank_df[["rerank_rank", "title", "vector_rank", "vector_score", "rerank_score", "content"]]
rerank_df


config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

,rerank_rank,title,vector_rank,vector_score,rerank_score,content
0,1,Batman Begins,5,0.504522,0.998291,蝙蝠俠的起源故事，風格沉穩黑暗，偏寫實，帶有訓練、成長與打擊犯罪元素。
1,2,黑暗騎士,4,0.481430,0.997861,蝙蝠俠與小丑的對決，黑暗、寫實、嚴肅的超級英雄動作片，節奏緊湊，充滿道德衝突。
2,3,蝙蝠俠大戰超人,2,0.412785,0.996955,蝙蝠俠與超人的衝突與對決，黑暗風格的超級英雄動作電影，氣氛嚴肅。
3,4,小丑,3,0.469989,0.995383,與蝙蝠俠宇宙有關的黑暗劇情片，角色心理描寫強烈，但不是典型超級英雄動作片。
4,5,復仇者聯盟,6,0.526243,0.991072,經典超級英雄動作片，英雄眾多、娛樂性高，但沒有蝙蝠俠，整體風格不算特別黑暗。
5,6,樂高蝙蝠俠電影,8,0.622011,0.989166,以蝙蝠俠為主角的動畫電影，雖然有英雄元素，但整體風格搞笑、歡樂、適合家庭。
6,7,守護者,1,0.390483,0.977783,黑暗、成人向、帶有政治與道德思辨的英雄電影，風格很沉重，但不是蝙蝠俠作品。
7,8,蜘蛛人：無家日,7,0.607120,0.924251,多重宇宙英雄冒險，屬於超級英雄動作片，但主角不是蝙蝠俠，風格較熱鬧。


## 並排比較：誰被升上來？誰被壓下去？

這一格是教學重點。  
請不要只看「第一名是誰」，而要看：

- 哪些候選在向量檢索時排名不錯，但 rerank 後掉下去
- 哪些候選本來不是最前面，但 rerank 後被拉上來

這正是 rerank 的價值。


In [10]:
compare_df = rerank_df.copy()
compare_df["rank_change"] = compare_df["vector_rank"] - compare_df["rerank_rank"]

def explain_change(x):
    if x > 0:
        return f"⬆️ 上升 {x}"
    elif x < 0:
        return f"⬇️ 下降 {abs(x)}"
    else:
        return "— 不變"

compare_df["變化"] = compare_df["rank_change"].apply(explain_change)
compare_df[["rerank_rank", "title", "vector_rank", "變化", "vector_score", "rerank_score"]]


,rerank_rank,title,vector_rank,變化,vector_score,rerank_score
0,1,Batman Begins,5,⬆️ 上升 4,0.504522,0.998291
1,2,黑暗騎士,4,⬆️ 上升 2,0.481430,0.997861
2,3,蝙蝠俠大戰超人,2,⬇️ 下降 1,0.412785,0.996955
3,4,小丑,3,⬇️ 下降 1,0.469989,0.995383
4,5,復仇者聯盟,6,⬆️ 上升 1,0.526243,0.991072
5,6,樂高蝙蝠俠電影,8,⬆️ 上升 2,0.622011,0.989166
6,7,守護者,1,⬇️ 下降 6,0.390483,0.977783
7,8,蜘蛛人：無家日,7,⬇️ 下降 1,0.607120,0.924251



### 若某部片被 rerank 往上拉
代表它不一定在向量空間裡最像，但從**完整語意匹配**來看，更符合使用者需求。

### 若某部片被 rerank 往下壓
通常代表它雖然含有關鍵字（例如「蝙蝠俠」或「超級英雄」），  
但在風格、語氣、限制條件上其實不夠符合。


In [11]:
# 看看最終 Top 3 候選
top_n = 3
top_docs = rerank_df.head(top_n)

for _, row in top_docs.iterrows():
    print(f"[Rerank Top {row['rerank_rank']}] {row['title']}")
    print(f"內容：{row['content']}")
    print("-" * 90)


[Rerank Top 1] Batman Begins
內容：蝙蝠俠的起源故事，風格沉穩黑暗，偏寫實，帶有訓練、成長與打擊犯罪元素。
------------------------------------------------------------------------------------------
[Rerank Top 2] 黑暗騎士
內容：蝙蝠俠與小丑的對決，黑暗、寫實、嚴肅的超級英雄動作片，節奏緊湊，充滿道德衝突。
------------------------------------------------------------------------------------------
[Rerank Top 3] 蝙蝠俠大戰超人
內容：蝙蝠俠與超人的衝突與對決，黑暗風格的超級英雄動作電影，氣氛嚴肅。
------------------------------------------------------------------------------------------


## 交給 LLaMA 生成最終推薦理由

現在我們不直接用向量檢索結果回答，  
而是把 **rerank 後的前幾名文件**交給 LLM，讓它產生更合理的推薦說明。

這就是典型的二階段 RAG：

1. 向量檢索先召回
2. rerank 重新排序
3. LLM 根據較乾淨的上下文回答


In [12]:
context = "\n\n".join(
    [f"標題：{row['title']}\n內容：{row['content']}" for _, row in top_docs.iterrows()]
)

prompt = ChatPromptTemplate.from_template("""
你是一位電影推薦助理。
請根據提供的候選電影資料，回答使用者的需求。

使用者需求：
{query}

候選資料：
{context}

請完成以下任務：
1. 先給出最推薦的 1~2 部電影
2. 清楚說明推薦理由
3. 順便指出哪些候選雖然相關，但不完全符合需求
4. 全程使用繁體中文回答
""")

chain = prompt | llm
response = chain.invoke({"query": query, "context": context})
print(response.content)


根據您的需求，我最推薦的兩部電影是《黑暗騎士》和《蝙蝠俠大戰超人》。

推薦理由是，這兩部電影都具有黑暗、嚴肅和寫實的風格，且都包含了英雄對決或打擊犯罪的元素。《黑暗騎士》中，蝙蝠俠與小丑的對決是整部電影的核心，充滿了道德衝突和緊湊的節奏。《蝙蝠俠大戰超人》則呈現了蝙蝠俠與超人的對決，氣氛嚴肅，黑暗風格的超級英雄動作電影。

至於《蝙蝠俠的起源故事》，雖然也具有黑暗和寫實的風格，但它更側重於蝙蝠俠的成長和訓練過程，雖然也包含了打擊犯罪的元素，但整體而言，對決和打擊犯罪的元素不如《黑暗騎士》和《蝙蝠俠大戰超人》那樣突出。

因此，如果您想要看一部黑暗、嚴肅、寫實的蝙蝠俠電影，且有英雄對決或打擊犯罪的感覺，我強烈推薦《黑暗騎士》和《蝙蝠俠大戰超人》。


## 再做一個「更極端」的查詢測試
  
我們再試一個更容易混淆的 query：

> 想看蝙蝠俠，但要適合小朋友、輕鬆一點

這時候理論上：
- `樂高蝙蝠俠電影` 會被 rerank 往前拉
- `黑暗騎士` 這類黑暗寫實作品可能會被壓下去


In [13]:
query2 = "我想看蝙蝠俠相關電影，但希望輕鬆、適合家庭觀看，小朋友也可以接受，不要太黑暗"

vector_results_q2 = vectorstore.similarity_search_with_score(query2, k=8)
candidate_docs_q2 = [doc for doc, _ in vector_results_q2]

rerank_inputs_q2 = [
    [query2, f"標題：{doc.metadata['title']}；內容：{doc.page_content}"]
    for doc in candidate_docs_q2
]
rerank_scores_q2 = reranker.predict(rerank_inputs_q2)

rows_q2 = []
for i, ((doc, score), rscore) in enumerate(zip(vector_results_q2, rerank_scores_q2), start=1):
    rows_q2.append({
        "title": doc.metadata["title"],
        "vector_rank": i,
        "vector_score": float(score),
        "rerank_score": float(rscore),
        "content": doc.page_content
    })

df_q2 = pd.DataFrame(rows_q2).sort_values("rerank_score", ascending=False).reset_index(drop=True)
df_q2["rerank_rank"] = np.arange(1, len(df_q2) + 1)
df_q2[["rerank_rank", "title", "vector_rank", "vector_score", "rerank_score"]]


,rerank_rank,title,vector_rank,vector_score,rerank_score
0,1,復仇者聯盟,2,0.640785,0.997850
1,2,樂高蝙蝠俠電影,1,0.515447,0.997557
2,3,蝙蝠俠大戰超人,7,0.802680,0.993784
3,4,Batman Begins,6,0.800810,0.993429
4,5,小丑,4,0.702559,0.993376
5,6,守護者,3,0.647913,0.990064
6,7,蜘蛛人：無家日,5,0.735517,0.800823
7,8,閃電俠,8,0.817498,0.799686


## 這個對照能說明什麼？

同樣一批資料，只是 query 改了，最適合的答案就可能不同。

這正是 rerank 的強項：

- 它不是只看「有沒有出現相似詞」
- 而是更接近在問：  
  **「這份候選內容，和這個 query 的完整需求，到底合不合？」**


## 總結

這份版本的重點不是只展示「有 rerank」，  
而是要讓學生清楚看到：

> **當候選資料彼此很像、query 條件又很細時，rerank 才是真正把 precision 拉起來的關鍵。**
